# Math

This section uses `sympy` to verify the correctness of the mathematical derivation. 

In [ ]:
import sympy
from sympy import Matrix, KroneckerProduct, conjugate
from sympy.physics.quantum import Dagger

# 开启高质量打印
sympy.init_printing(use_unicode=True)

In [ ]:
# --- 1. 定义符号和基本矩阵 ---

# 量子态系数

alpha, beta = sympy.symbols("alpha beta", complex=True)
alpha_c, beta_c = sympy.symbols(
    "alpha_c beta_c", complex=True
)  # 手动定义共轭，便于替换

# 误差参数 (小量)
eps_C, eps_X = sympy.symbols("epsilon_C epsilon_X", real=True, positive=True)
a, b, c, d = sympy.symbols("a b c d", real=True, positive=True)

# 泡利矩阵和单位阵
I2 = Matrix([[1, 0], [0, 1]])
X = Matrix([[0, 1], [1, 0]])
I4 = KroneckerProduct(I2, I2)

# CNOT 门 (第1个比特为控制，第2个为目标)
CNOT = Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

In [ ]:
# --- 2. 初始状态和理想编码 ---

# 初始密度矩阵 (Eq. 4.3)
rho_target = Matrix(
    [[alpha * alpha_c, alpha * beta_c], [beta * alpha_c, beta * beta_c]]
)  # |psi><psi| where |psi> = alpha|0> + beta|1>
rho_ancilla = Matrix([[1, 0], [0, 0]])  # |0><0|
rho_initial = KroneckerProduct(rho_target, rho_ancilla)

# 编码操作U_enc (CNOT, then X on ancilla)
U_enc = KroneckerProduct(I2, X) * CNOT
psi_initial_vec = KroneckerProduct(Matrix([alpha, beta]), Matrix([1, 0]))
psi_encoded_vec = U_enc * psi_initial_vec

# 理想编码后的密度矩阵
rho_encoded_ideal = psi_encoded_vec * Dagger(psi_encoded_vec)
# 替换 alpha*conjugate(alpha) -> abs(alpha)**2
rho_encoded_ideal = rho_encoded_ideal.subs(
    {
        alpha * conjugate(alpha): sympy.Abs(alpha) ** 2,
        beta * conjugate(beta): sympy.Abs(beta) ** 2,
        alpha * conjugate(beta): alpha * beta_c,
        beta * conjugate(alpha): beta * alpha_c,
    }
)

rho_encoded_ideal

In [ ]:
def remove_high_order_terms(expr, symbols):
    """移除表达式中的高阶小量项"""

    eps = sympy.Symbol("epsilon", real=True, positive=True)

    expr_with_eps = expr.subs({sym: eps * sym for sym in symbols})

    linear_expr_with_eps = expr_with_eps.series(eps, 0, 2).removeO()

    final_expr = linear_expr_with_eps.subs({eps: 1})
    return final_expr

In [ ]:
# --- 3. 含噪量子门的推导 (验证 Eq. 9) ---


def enc_with_noise(rho):
    rho1 = (1 - eps_C) * CNOT * rho * Dagger(CNOT) + eps_C * (I4 / 4)
    rho2 = (1 - eps_X) * KroneckerProduct(I2, X) * rho1 * Dagger(
        KroneckerProduct(I2, X)
    ) + eps_X * (I4 / 4)
    return rho2


rho_encoded_noisy = enc_with_noise(rho_initial)
rho_encoded_noisy

In [ ]:
# 提取对角线元素，即概率分布 P (验证 Eq. 10 的正确形式)
P_vector = sympy.zeros(4, 1)
for i in range(4):
    P_vector[i] = rho_encoded_noisy[i, i]

P_vector

In [ ]:
# 一阶近似：将 eps_C^2, eps_X^2, eps_C*eps_X 等高阶小量忽略
P_vector_approx = P_vector
for i in range(4):
    P_vector_approx[i] = remove_high_order_terms(P_vector[i], [eps_C, eps_X])

P_vector_approx = P_vector_approx.subs(
    {
        alpha * alpha_c: sympy.Abs(alpha) ** 2,
        beta * beta_c: sympy.Abs(beta) ** 2,
        alpha * conjugate(beta): alpha * beta_c,
        beta * conjugate(alpha): beta * alpha_c,
    }
)

A = sympy.Symbol("A", real=True, positive=True)
P_vector_approx = P_vector_approx.subs(
    {
        sympy.Abs(alpha) ** 2: A,
        sympy.Abs(beta) ** 2: 1 - A,
    }
)

for i in range(4):
    P_vector_approx[i] = sympy.collect(P_vector_approx[i].expand(), A)

eps = sympy.Symbol("e", real=True, positive=True)
P_vector_approx = P_vector_approx.subs({eps_X: eps - eps_C})

P_vector_approx

In [ ]:
# --- 4. 考虑读出噪声 (验证 Eq. 17) ---

# 单比特读出噪声矩阵 (Eq. 14, 15)
M1 = Matrix([[1 - a, b], [a, 1 - b]])
M2 = Matrix([[1 - c, d], [c, 1 - d]])

# 总读出混淆矩阵 (Eq. 16.1)
M_readout = KroneckerProduct(M1, M2)

# 施加读出噪声后的概率分布 P' (Eq. 16.2)
P_prime_exact = sympy.Matrix(M_readout * P_vector)

# simplify
P_prime_exact = P_prime_exact.subs(
    {
        alpha * alpha_c: sympy.Abs(alpha) ** 2,
        beta * beta_c: sympy.Abs(beta) ** 2,
    }
)

P_prime_exact

In [ ]:
# 对 P' 进行一阶线性化
# small_params 是所有的小量
small_params = [eps_C, eps_X, a, b, c, d]
P_prime_linear = sympy.zeros(4, 1)
for i in range(4):
    # series在多变量时行为复杂，手动展开并只保留一阶项更可靠
    expr = sympy.expand(P_prime_exact[i])
    tmp = remove_high_order_terms(expr, small_params)

    # simplify
    tmp = sympy.factor(tmp)
    tmp = tmp.subs({eps_X + eps_C: eps})
    tmp = sympy.collect(tmp, [sympy.Abs(alpha) ** 2, sympy.Abs(beta) ** 2])
    tmp = tmp.subs({eps_X + eps_C: eps})

    P_prime_linear[i] = tmp

P_prime_linear

In [ ]:
P_prime_exact.subs(
    {
        eps_C: 0,
        eps_X: 0,
        alpha * alpha_c: sympy.Abs(alpha) ** 2,
        beta * beta_c: 1 - sympy.Abs(alpha) ** 2,
        c: a,
        d: b,
    }
)

In [ ]:
P_L_exact_unnormalized = Matrix([P_prime_exact[1], P_prime_exact[2]])
tmp = sympy.simplify(
    P_L_exact_unnormalized.subs(
        {
            eps_C: 0,
            eps_X: 0,
            alpha * alpha_c: sympy.Abs(alpha) ** 2,
            beta * beta_c: 1 - sympy.Abs(alpha) ** 2,
            c: a,
            d: b,
        }
    )
)
tmp

# sympy.simplify(sum(tmp))

In [ ]:
# --- 5. 归一化和最终逻辑概率 (验证 Eq. 26.3) ---

# 提取 |01> 和 |10> 的概率 (未归一化)
P_L_unnormalized = Matrix([P_prime_linear[1], P_prime_linear[2]])
P_L_unnormalized = P_L_unnormalized.subs(
    {sympy.Abs(beta) ** 2: 1 - sympy.Abs(alpha) ** 2}
)

P_L_unnormalized

In [ ]:
# 计算归一化因子 S (Eq. 19)
S = P_L_unnormalized[0] + P_L_unnormalized[1]
S = sympy.expand(S)
S = sympy.collect(S, sympy.Abs(alpha) ** 2)
S

In [ ]:
# 定义 T (Eq. 21.1)
T = 1 - S

# 归一化后的概率 P'_L = P_L / S ≈ P_L * (1 + T)
P_L_final = sympy.zeros(2, 1)
for i in range(2):
    expr = P_L_unnormalized[i] * (1 + T)

    # remove high-order terms
    expr = sympy.expand(expr)
    expr = remove_high_order_terms(expr, [a, b, c, d, eps])

    # simplify
    expr = sympy.collect(expr, [sympy.Abs(alpha)])
    P_L_final[i] = expr


P_L_final

In [ ]:
sympy.simplify(sum(P_L_final))

In [ ]:
# 在平衡读出误差情况下的逻辑概率: a + d = b + c
P_L_balanced = P_L_final.subs({a + d - b - c: 0})

P_L_balanced

In [ ]:
P_ideal = Matrix([sympy.Abs(alpha) ** 2, 1 - sympy.Abs(alpha) ** 2])

P_measure = M1 * P_ideal
P_measure

# Simulate

This section validates the effectiveness of the encoding through code simulation.

In [ ]:
import numpy as np
from numpy.typing import NDArray


def measurement_decode(
    data: NDArray[np.int_], enc: NDArray[np.int_]
) -> NDArray[np.int_]:
    assert data.shape == enc.shape

    # (data[i], enc[i]) == (0, 1) --> decode to 0
    # (data[i], enc[i]) == (1, 0) --> decode to 1
    # Other combinations are errors and ignored
    result: list[int] = []
    for d, e in zip(data, enc):
        if (d, e) == (0, 1):
            result.append(0)
        elif (d, e) == (1, 0):
            result.append(1)
        else:
            result.append(-1)  # Indicate error with -1
    return np.array(result)


measurement_decode(np.array([0, 1, 1, 0, 0]), np.array([1, 1, 0, 0, 1]))

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Sampler
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error


noise_model = NoiseModel()

# readout error
prob_meas1_prep0 = 0.01
prob_meas0_prep1 = 0.03
readout_error = ReadoutError(
    [[1 - prob_meas1_prep0, prob_meas1_prep0], [prob_meas0_prep1, 1 - prob_meas0_prep1]]
)
noise_model.add_all_qubit_readout_error(readout_error)

sampler_without_gates_error = Sampler(mode=AerSimulator(noise_model=noise_model))

In [ ]:
noise_model = NoiseModel()

# readout error
noise_model.add_all_qubit_readout_error(readout_error)

# single gate depolarizing error
error_1q = 1.0 - 0.9992
noise_model.add_all_qubit_quantum_error(depolarizing_error(error_1q, 1), ["x"])

# two-qubit gate depolarizing error
error_2q = 1.0 - 0.992
noise_model.add_all_qubit_quantum_error(depolarizing_error(error_2q, 2), ["cx"])

sampler = Sampler(mode=AerSimulator(noise_model=noise_model))

In [ ]:
sampler_idel = Sampler(mode=AerSimulator())

Encoding circuit:

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.circuit import Parameter


circ = QuantumCircuit()

qreg = QuantumRegister(1, "Q")
qreg_anc = QuantumRegister(1, "Q_anc")
creg = ClassicalRegister(1, "C")
creg_anc = ClassicalRegister(1, "C_anc")

circ.add_register(qreg)
circ.add_register(qreg_anc)
circ.add_register(creg)
circ.add_register(creg_anc)

# prepare state
circ.ry(Parameter("theta"), qreg[0])

# measurement encoding
circ.cx(qreg[0], qreg_anc[0])
circ.x(qreg_anc[0])

circ.measure(qreg[0], creg[0])
circ.measure(qreg_anc[0], creg_anc[0])

circ.draw("mpl")

In [ ]:
circ_non_enc = QuantumCircuit(1, 1)
circ_non_enc.ry(Parameter("theta"), 0)
circ_non_enc.measure(0, 0)

circ_non_enc.draw("mpl")

In [ ]:
import numpy as np

thetas = np.linspace(0, np.pi, 50)

num_shots = 100_000

Ideal measurement probability:

In [ ]:
# result_ideal = sampler_idel.run([(circ_non_enc, theta) for theta in thetas], shots=1000).result()

# prob0_ideal: list[float] = []

# for i, theta in enumerate(thetas):
#     # print(f"Theta: {theta:.2f}")
#     arr = result_ideal[i].data.c.array
#     count_dict: dict[int, int] = {
#         0: int(np.sum(arr == 0)),
#         1: int(np.sum(arr == 1))
#     }
#     # print(f"Ideal counts: {count_dict}")
#     normalized_counts = count_dict[0] + count_dict[1]
#     # print(f"p0 = {count_dict[0] / 1000:.3f}, p1 = {count_dict[1] / 1000:.3f}\n")

#     prob0_ideal.append(count_dict[0] / normalized_counts)

prob0_ideal: list[float] = []
for i, theta in enumerate(thetas):
    prob0_ideal.append(np.cos(theta / 2) ** 2)

Probability of the encoding:

In [ ]:
result = sampler.run([(circ, theta) for theta in thetas], shots=num_shots).result()

prob0_enc: list[float] = []
prob_invalid_enc: list[float] = []

for i, theta in enumerate(thetas):
    c = result[i].data.C.array
    c_anc = result[i].data.C_anc.array
    decode = measurement_decode(c, c_anc)

    counts: dict[int, int] = {
        0: int(np.sum(decode == 0)),
        1: int(np.sum(decode == 1)),
        -1: int(np.sum(decode == -1)),
    }

    normalized_counts = counts[0] + counts[1]
    tmp = counts[0] / normalized_counts

    prob0_enc.append(tmp)

    prob_invalid_enc.append(counts.get(-1, 0) / sum(counts.values()))

Probability of non-encoding:

In [ ]:
result_non_enc = sampler.run(
    [(circ_non_enc, theta) for theta in thetas], shots=num_shots
).result()

prob0_non_enc: list[float] = []

for i, theta in enumerate(thetas):
    arr: NDArray[np.int_] = result_non_enc[i].data.c.array
    decode = arr

    count_dict: dict[int, int] = {
        0: int(np.sum(decode == 0)),
        1: int(np.sum(decode == 1)),
    }
    normalized_counts = count_dict[0] + count_dict[1]
    tmp = count_dict[0] / normalized_counts

    prob0_non_enc.append(tmp)

Probability of encoding but without considering error of gates:

In [ ]:
result_enc_without_gates_error = sampler_without_gates_error.run(
    [(circ, theta) for theta in thetas], shots=num_shots
).result()

prob0_enc_without_gates_error: list[float] = []
prob_invalid_enc_without_gates_error: list[float] = []

for i, theta in enumerate(thetas):
    c = result_enc_without_gates_error[i].data.C.array
    c_anc = result_enc_without_gates_error[i].data.C_anc.array
    decode = measurement_decode(c, c_anc)

    counts: dict[int, int] = {
        0: int(np.sum(decode == 0)),
        1: int(np.sum(decode == 1)),
        -1: int(np.sum(decode == -1)),
    }

    normalized_counts = counts[0] + counts[1]
    tmp = counts[0] / normalized_counts

    prob0_enc_without_gates_error.append(tmp)

    prob_invalid_enc_without_gates_error.append(counts.get(-1, 0) / sum(counts.values()))

Plot three kinds of probabilities:
- Ideal probability
- Encoding
- Non-encoding

In [ ]:
import matplotlib.pyplot as plt


# plot three kinds of probabilities
fig, ax = plt.subplots(figsize=(8, 6))

x = thetas / np.pi

ax.plot(x, prob0_ideal, "o-", ms=4, label="Ideal (no noise)", color="black")
ax.plot(
    x, prob0_non_enc, "s--", ms=4, label="Non-encoded (with noise)", color="red"
)
ax.plot(x, prob0_enc, "^--", ms=4, label="Encoded (with noise)", color="blue")

ax.legend()

ax.set_xlabel("Theta / π")
ax.set_ylabel("Probability of measuring 0")

plt.show()

Using TVD (total variation distance) to measure the distance between measurement probablities and the ideal probability.

In [ ]:
def calc_tvd(p: NDArray[np.floating], q: NDArray[np.floating]) -> float:
    assert p.shape == q.shape

    return 0.5 * np.sum(np.abs(p - q))


tvd_ideal_non_enc = [
    calc_tvd(np.array([p, 1 - p]), np.array([q, 1 - q]))
    for p, q in zip(prob0_ideal, prob0_non_enc)
]

tvd_ideal_enc = [
    calc_tvd(np.array([p, 1 - p]), np.array([q, 1 - q]))
    for p, q in zip(prob0_ideal, prob0_enc)
]

tvd_ideal_enc_without_gates_error = [
    calc_tvd(np.array([p, 1 - p]), np.array([q, 1 - q]))
    for p, q in zip(prob0_ideal, prob0_enc_without_gates_error)
]

x = thetas / np.pi

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    x,
    tvd_ideal_non_enc,
    "s--",
    ms=4,
    label="TVD: Ideal vs Non-encoded",
    color="red",
)
ax.plot(x, tvd_ideal_enc, "^--", ms=4, label="TVD: Ideal vs Encoded", color="blue")

ax.plot(
    x,
    tvd_ideal_enc_without_gates_error,
    "d--",
    ms=4,
    label="TVD: Ideal vs Encoded (w/o gate errors)",
    color="green",
)

ax.legend()

ax.set_xlabel("Theta / π")
ax.set_ylabel("Total Variation Distance (TVD)")

plt.title(f"TVD between Ideal and Noisy Measurements ({num_shots:,} shots)")
plt.show()

Plot the frequency of invalid measurement results:

In [ ]:
# plot the frequency of invalid measurement results:

from turtle import color


fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(
    x,
    prob_invalid_enc,
    "^--",
    ms=6,
    label="Encoded (with gate errors)",
    color="blue",
)
mean_invalid_enc = np.mean(prob_invalid_enc)
ax.axhline(
    float(mean_invalid_enc),
    color="blue",
    linestyle=":",
)

ax.plot(
    x,
    prob_invalid_enc_without_gates_error,
    "s--",
    ms=6,
    label="Encoded (without gate errors)",
    color="green",
)
mean_invalid_enc_without_gates_error = np.mean(prob_invalid_enc_without_gates_error)
ax.axhline(
    float(mean_invalid_enc_without_gates_error),
    color="green",
    linestyle=":",
)

ax.set_xlabel("Theta / π")
ax.set_ylabel("Frequency of Invalid Measurement Results")

ax.legend(loc="upper right")
plt.title(f"Frequency of Invalid Measurement Results ({num_shots:,} shots)")

plt.show()